
# Семинар: визуализация данных в Python

Ноутбук основан на материалах лекции: базовые возможности Matplotlib, оформление осей, различие между `pyplot` и объектно-ориентированным (ОО) подходом, цвета и цветовые карты, несколько систем координат на одном рисунке, разные типы графиков, визуализация средствами Pandas и разведочный анализ данных с Seaborn.

## Правила
1. В начале ноутбука задаётся `STUDENT_ID`.
2. По `STUDENT_ID` генерируются **индивидуальные исходные данные** и локальные файлы.

## Структура
- Первая пара: задания 1–10.
- Вторая пара: задания 11–20.


## Представляемые результаты
1. Все графики должны иметь заголовок, подписи осей и читаемую легенду там, где она нужна.
2. В заданиях 2, 5, 7, 9, 12, 15, 18 и 20 нужно добавить краткий аналитический вывод на 2–4 предложения.
3. Не удаляй генерацию данных и не подменяй набор данных внешним CSV.
4. Код должен быть воспроизводимым: повторный запуск ноутбука не должен приводить к ошибкам.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 9
sns.set_theme(style="whitegrid", font="DejaVu Sans")

def make_transport_dataset(student_variant: int = 7) -> pd.DataFrame:
    import math
    rng = np.random.default_rng(2024 + student_variant * 17)
    dates = pd.date_range("2024-02-01", periods=84, freq="D")
    shifts = ["утро", "день", "вечер"]
    route_types = ["кампус", "центр", "спальный"]
    weather_states = ["ясно", "облачно", "дождь", "снег"]
    weekday_order = ["понедельник", "вторник", "среда", "четверг", "пятница", "суббота", "воскресенье"]

    rows = []
    route_distance = {"кампус": 8, "центр": 14, "спальный": 11}
    shift_factor = {"утро": 1.20, "день": 0.92, "вечер": 1.08}
    weather_penalty = {"ясно": 0.0, "облачно": 1.2, "дождь": 4.5, "снег": 8.0}
    weather_load = {"ясно": 0, "облачно": 8, "дождь": 18, "снег": 26}
    weather_energy = {"ясно": 0.0, "облачно": 1.5, "дождь": 3.5, "снег": 5.0}
    weather_satisfaction = {"ясно": 0.30, "облачно": 0.05, "дождь": -0.25, "снег": -0.45}

    for i, date in enumerate(dates):
        dow = weekday_order[date.weekday()]
        weekend = dow in ["суббота", "воскресенье"]
        season = math.sin((i + 1 + student_variant) / 9)
        base_temp = 7 + 9 * season + (student_variant - 5) * 0.2
        for shift_idx, shift in enumerate(shifts):
            route = route_types[(i + shift_idx + student_variant) % len(route_types)]
            if weekend and shift == "утро":
                route = "спальный"
            if (not weekend) and shift == "утро" and i % 5 == 0:
                route = "кампус"

            weather_noise = rng.choice(weather_states, p=[0.32, 0.31, 0.24, 0.13])
            if base_temp > 10 and weather_noise == "снег":
                weather = "дождь"
            elif base_temp < -1 and weather_noise == "дождь":
                weather = "снег"
            else:
                weather = weather_noise

            temperature = base_temp + {"утро": -2.0, "день": 1.5, "вечер": -0.5}[shift] + rng.normal(0, 2.2)
            humidity = 58 + {"ясно": -8, "облачно": 4, "дождь": 18, "снег": 15}[weather] + rng.normal(0, 7)
            humidity = float(np.clip(humidity, 25, 100))
            wind = 2.5 + {"ясно": 0.3, "облачно": 0.9, "дождь": 1.8, "снег": 2.2}[weather] + rng.normal(0, 0.8)
            wind = float(np.clip(wind, 0.2, 11.5))

            base_passengers = {"кампус": 120, "центр": 160, "спальный": 110}[route]
            weekday_bonus = 18 if dow in ["понедельник", "вторник", "среда", "четверг"] else 8 if dow == "пятница" else -28
            rush_bonus = 22 if shift == "утро" and not weekend else 15 if shift == "вечер" and not weekend else -5
            weather_effect = weather_load[weather]
            temp_effect = -1.8 * abs(temperature - 16)
            passengers = base_passengers * shift_factor[shift] + weekday_bonus + rush_bonus - weather_effect + temp_effect + rng.normal(0, 11)
            passengers = int(np.clip(round(passengers), 35, 260))

            incidents_lambda = 0.10 + (0.0015 * passengers) + {"ясно": 0.0, "облачно": 0.08, "дождь": 0.18, "снег": 0.25}[weather]
            incidents = int(rng.poisson(incidents_lambda))
            incidents = min(incidents, 5)

            delay = 1.8 + 0.055 * passengers + weather_penalty[weather] + 2.2 * incidents + 0.18 * wind + rng.normal(0, 2.5)
            delay = float(np.clip(delay, 0, 35))

            occupancy = 32 + passengers / 2.1 + {"кампус": 4, "центр": 9, "спальный": -2}[route] + rng.normal(0, 6)
            occupancy = float(np.clip(occupancy, 20, 100))

            energy = 18 + 0.18 * passengers + 1.05 * route_distance[route] + 0.42 * abs(temperature - 18) + weather_energy[weather] + 0.8 * incidents + rng.normal(0, 3.2)
            energy = float(np.clip(energy, 20, 95))

            ticket_income = 42 + 1.55 * passengers + {"кампус": -12, "центр": 28, "спальный": 5}[route] + rng.normal(0, 22)
            ticket_income = float(np.clip(ticket_income, 80, 520))

            satisfaction = 4.8 - 0.045 * delay - 0.012 * max(occupancy - 75, 0) + weather_satisfaction[weather] + rng.normal(0, 0.18)
            satisfaction = float(np.clip(satisfaction, 1.2, 5.0))

            rows.append({
                "record_id": f"R{student_variant:02d}-{i:03d}-{shift_idx + 1}",
                "date": date,
                "weekday": dow,
                "shift": shift,
                "route_type": route,
                "weather": weather,
                "temperature_c": round(float(temperature), 2),
                "humidity_pct": round(humidity, 2),
                "wind_mps": round(wind, 2),
                "passengers": passengers,
                "delay_min": round(delay, 2),
                "incidents": incidents,
                "energy_kwh": round(energy, 2),
                "occupancy_pct": round(occupancy, 2),
                "ticket_income_kzt": round(ticket_income, 2),
                "satisfaction_score": round(satisfaction, 2),
            })

    df = pd.DataFrame(rows)
    df["weekday"] = pd.Categorical(df["weekday"], categories=weekday_order, ordered=True)
    df["shift"] = pd.Categorical(df["shift"], categories=shifts, ordered=True)
    df["route_type"] = pd.Categorical(df["route_type"], categories=route_types, ordered=True)
    df["weather"] = pd.Categorical(df["weather"], categories=weather_states, ordered=True)
    df["week"] = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month.astype(int)
    df["peak_load"] = df["occupancy_pct"] >= np.percentile(df["occupancy_pct"], 75)

    nan_idx = rng.choice(df.index, size=6, replace=False)
    df.loc[nan_idx[:3], "satisfaction_score"] = np.nan
    df.loc[nan_idx[3:], "ticket_income_kzt"] = np.nan
    return df

def show_basic_info(df: pd.DataFrame) -> None:
    display(df.head())
    display(df.describe(include="all").T[["count", "unique", "mean", "std", "min", "max"]])

# Впишите свой идентификатор свой идентификатор.
STUDENT_ID = "123456"  # например: "123456" из "123456@edu.fa.ru"
STUDENT_ID = int(STUDENT_ID)
df = make_transport_dataset(STUDENT_ID)
show_basic_info(df)
print(f"Размер набора данных: {df.shape}")

## Пара 1. Matplotlib: базовые объекты, оформление, компоновка, типы графиков

### Задание 1. Классификация признаков и выбор визуализации

Составь таблицу минимум для 12 признаков набора данных. Для каждого признака укажи:
1) тип признака (`количественный` / `качественный`),
2) подтип (`непрерывный`, `дискретный`, `номинальный`, `порядковый`),
3) один предпочтительный способ визуализации,
4) краткое обоснование выбора.

**Что должно получиться:**
- результат оформлен в виде `DataFrame`
- в таблице есть не менее 12 строк
- для каждого признака есть обоснование из 1 короткой фразы


In [ ]:
# Создай таблицу feature_review

### Задание 2. Две шкалы на одном рисунке

Построй **объектно-ориентированным интерфейсом Matplotlib** график среднесуточного числа пассажиров и среднесуточной задержки.
Используй `twinx()`: на левой оси — `avg_passengers`, на правой — `avg_delay`.
Отдельно выведи даты максимума по каждому показателю.

**Что должно получиться:**
- использован интерфейс `fig, ax = plt.subplots()`
- на рисунке есть 2 оси Y
- легенда объединяет обе линии
- после графика напечатаны 2 даты и 2 значения


In [ ]:
# Агрегируй данные по дате и построй график

**Краткий вывод:**

*Заполни 2–4 предложениями по своему графику.*

### Задание 3. Исправление ошибочного кода: pyplot vs OO

Ниже дан фрагмент кода. Он написан неудачно и содержит концептуальные ошибки.
Перепиши его так, чтобы он корректно работал через объектно-ориентированный интерфейс.
Затем перечисли минимум 3 проблемы исходного варианта.

```python
daily = df.groupby('date', as_index=False).agg(avg_passengers=('passengers', 'mean'), avg_delay=('delay_min', 'mean'))
plt.figure(figsize=(12, 4))
plt.plot(daily['date'], daily['avg_passengers'], color='tab:blue', label='Пассажиры')
ax2 = ax.twinx()
plt.plot(daily['date'], daily['avg_delay'], color='tab:red', label='Задержка')
plt.legend(loc='upper left')
plt.title('Плохой пример')
```

**Что должно получиться:**
- новый код работает без ошибок
- использован OO-подход
- описаны минимум 3 проблемы исходного фрагмента


In [ ]:
# Перепиши код

**Какие ошибки были в исходном варианте?**

1.
2.
3.


### Задание 4. Топ-15 наблюдений по пассажиропотоку

Найди 15 наблюдений с максимальным значением `passengers`. Построй для них столбчатую диаграмму.
По оси X выведи `record_id`, поверни подписи, задай явные границы по оси Y и подпиши **три** самых больших столбца их значениями.

**Что должно получиться:**
- взяты именно 15 записей
- есть подписи для 3 максимальных столбцов
- ось X оформлена так, чтобы подписи читались


In [ ]:
# Построй столбчатую диаграмму для top-15

### Задание 5. Стили линий, маркеры и пользовательские цвета

Сгруппируй данные по `weekday` и `route_type`, вычисли среднее число пассажиров.
На одном графике построй три линии: для `кампус`, `центр`, `спальный`.
Для каждой линии используй **разный** стиль линии, маркер и цвет (цвет задай через hex-код или именованный цвет, а не только через стандартный цикл).
В выводе напиши, какой тип маршрута сильнее всего проседает на выходных.

**Что должно получиться:**
- на одном графике три линии
- для линий различаются цвет, маркер и стиль
- есть краткий текстовый вывод


In [ ]:
# Построй график по weekday и route_type

**Краткий вывод:**

*Заполни 2–3 предложениями.*

### Задание 6. Композиция 2x2 из разных графиков Matplotlib

Собери рисунок 2x2 с помощью `plt.subplots(...)`. Включи в него:
1) гистограмму `temperature_c`,
2) точечную диаграмму `humidity_pct` vs `wind_mps`,
3) столбчатую диаграмму среднего числа инцидентов по погоде,
4) линейный график среднего `delay_min` по сменам.
Используй `layout='constrained'` или аналогичную настройку аккуратной компоновки.

**Что должно получиться:**
- все 4 подграфика размещены на одном рисунке
- у каждого подграфика свой заголовок
- рисунок читаемый без наложения подписей


In [ ]:
# Собери рисунок 2x2

### Задание 7. Точечная диаграмма с кодированием размера и цвета

Построй `scatter`, где:
- по оси X — `temperature_c`,
- по оси Y — `passengers`,
- размер маркера зависит от `delay_min`,
- цвет зависит от `energy_kwh`.
Добавь цветовую шкалу и подпиши точку с максимальным `energy_kwh`.

**Что должно получиться:**
- размер и цвет кодируют разные признаки
- добавлена `colorbar`
- подписана точка максимального энергопотребления


In [ ]:
# Построй scatter с размером и цветом

**Краткий вывод:**

*Заполни 2–4 предложениями.*

### Задание 8. Сгруппированная столбчатая диаграмма

Построй сгруппированную столбчатую диаграмму среднего числа инцидентов по дням недели отдельно для каждого `route_type`.
Расположи группы столбцов корректно вручную через смещения по оси X. Легенду вынеси в удобное место.

**Что должно получиться:**
- для каждого дня недели есть 3 столбца
- смещения рассчитаны вручную
- подписи дней недели читаются


In [ ]:
# Построй grouped bar chart

### Задание 9. Тепловая карта загрузки по дню недели и смене

Построй тепловую карту среднего числа пассажиров по комбинациям `weekday` × `shift`.
Можно использовать `imshow` или `sns.heatmap`, но подписи осей и цветовая шкала обязательны.
Подбери цветовую карту, у которой изменение яркости хорошо согласуется с ростом величины.

**Что должно получиться:**
- использована матрица из сводной таблицы
- на графике есть цветовая шкала
- в тексте объяснён выбор цветовой карты


In [ ]:
# Построй heatmap для weekday x shift

**Почему выбрана именно эта цветовая карта?**

*Кратко объясни.*

### Задание 10. Контурный и трёхмерный график функции двух переменных

Задай вариантную функцию двух переменных:
`f(x, y) = sin((variant + 1) * x / 3) * cos(y / 2) + 0.08 * (x - 5)**2 + 0.03 * (y - 6)**2`
на прямоугольной сетке `x in [0, 10]`, `y in [0, 12]`.
Построй два графика: линии уровня (`contour`) и поверхность (`plot_surface`).

**Что должно получиться:**
- использован `meshgrid`
- построены оба графика
- в 3D-графике применена цветовая карта


In [ ]:
# Построй contour и 3D surface

## Пара 2. Pandas и Seaborn: быстрые графики и разведочный анализ данных

### Задание 11. Быстрые графики средствами Pandas

С помощью `DataFrame.plot(...)` собери строку из трёх графиков:
1) `temperature_c` vs `passengers`,
2) `passengers` vs `ticket_income_kzt`,
3) гистограмма `delay_min`.
Не используй напрямую `plt.scatter(...)` и `plt.hist(...)` в этом задании.

**Что должно получиться:**
- используется именно интерфейс Pandas
- получаются 3 разных графика в одной строке


In [ ]:
# Используй DataFrame.plot для трёх графиков

### Задание 12. Гистограмма распределения с разной шириной корзины

Построй две гистограммы `delay_min` через `sns.displot`: с `binwidth=1.5` и `binwidth=3.0`.
Сравни, как меняется визуальное восприятие распределения, и посчитай коэффициент асимметрии признака.

**Что должно получиться:**
- построены обе гистограммы
- вычислена асимметрия
- написано сравнение двух способов разбиения


In [ ]:
# Построй displot и вычисли skewness

**Краткий вывод:**

*Заполни 2–4 предложениями.*

### Задание 13. KDE для подвыборки и ограничение метода

Выдели подвыборку `df_center`, где `route_type == 'центр'`. Построй KDE-распределения `passengers` по сменам (`hue='shift'`).
После графика напиши, в каких случаях такой график может вводить в заблуждение.

**Что должно получиться:**
- использована подвыборка `центр`
- KDE построено по сменам
- в тексте указано ограничение метода


In [ ]:
# Построй KDE для df_center

**Когда KDE может искажать впечатление о данных?**

*Заполни 2–3 предложениями.*

### Задание 14. Создание нового признака и распределение с hue

Создай бинарный признак `delay_group`: `'высокая задержка'`, если `delay_min` не меньше 75-го процентиля, иначе `'обычная'`.
Построй распределение `passengers` с `hue='delay_group'` и сравни медианы по двум группам.

**Что должно получиться:**
- новый признак создан программно
- на графике группы различимы
- медианы посчитаны и сопоставлены


In [ ]:
# Создай delay_group и построй распределение

### Задание 15. Boxplot и программный поиск выбросов

Построй `boxplot` для `delay_min` по `weather`.
Затем найди выбросы по правилу IQR **в коде** и выведи таблицу с полями `record_id`, `date`, `weather`, `route_type`, `delay_min`.
Сравни найденные записи с визуальной картиной boxplot.

**Что должно получиться:**
- boxplot построен
- выбросы найдены программно
- показана таблица выбросов
- есть краткий аналитический вывод


In [ ]:
# Построй boxplot и найди выбросы через IQR

**Краткий вывод:**

*Заполни 2–4 предложениями.*

### Задание 16. Violinplot со split по бинарному признаку

Построй `violinplot` для `delay_min` по `shift` с разбиением по `peak_load` (`split=True`).
Сделай вывод, в каких сменах высокая загрузка сильнее всего связана с увеличением задержки.

**Что должно получиться:**
- использован `split=True`
- график содержит сравнение двух состояний `peak_load`


In [ ]:
# Построй violinplot

### Задание 17. Pairplot для числовых признаков

Возьми случайную подвыборку из 120 строк: `df.sample(120, random_state=STUDENT_ID)`.
Построй `pairplot` для признаков `temperature_c`, `passengers`, `delay_min`, `energy_kwh`, `satisfaction_score` с раскраской по `route_type`.

**Что должно получиться:**
- использована подвыборка из 120 строк
- в pairplot включено 5 числовых признаков


In [ ]:
# Построй pairplot

### Задание 18. Матрица корреляций и интерпретация связей

Построй тепловую карту корреляций для числовых признаков.
Найди:
1) наиболее сильную **положительную** интерпретируемую связь,
2) наиболее сильную **отрицательную** интерпретируемую связь.
Пары `week`–`month` и аналогичные календарные дубликаты не использовать.

**Что должно получиться:**
- построена heatmap с подписями коэффициентов
- указаны 2 пары признаков
- даны краткие пояснения


In [ ]:
# Построй heatmap корреляций и найди пары

**Краткий вывод:**

*Заполни 2–4 предложениями.*

### Задание 19. Jointplot и простая линейная аппроксимация

Для маршрута `центр` построй `jointplot(kind='reg')` для признаков `passengers` и `energy_kwh`.
Отдельно оцени коэффициенты линейной аппроксимации через `np.polyfit` и интерпретируй коэффициент наклона.

**Что должно получиться:**
- график построен по подвыборке `центр`
- коэффициенты аппроксимации вычислены


In [ ]:
# Построй jointplot и оцени линейную модель

### Задание 20. Мини-дашборд и краткая аналитическая записка

Собери итоговый дашборд 2x2 для диспетчера перевозок. Он должен отвечать минимум на 3 вопроса:
1) когда наблюдается наибольшая загрузка,
2) при какой погоде задержки наиболее проблемны,
3) что сильнее всего связано с удовлетворённостью.
Под графиками или после них напиши аналитическую записку на 5–7 предложений с конкретными наблюдениями из данных.

**Что должно получиться:**
- в дашборде 4 графика
- есть связный текстовый вывод на 5–7 предложений
- в тексте есть ссылки на конкретные закономерности из данных


In [ ]:
# Собери итоговый дашборд

**Аналитическая записка:**

*Заполни 5–7 предложениями.*